# 01. Data Understanding & Dataset Exploration
## Intelligent Cyber Threat Detection & Network Traffic Analysis
**Benchmark Dataset:** CIC-IDS2017 (Canadian Institute for Cybersecurity, UNB)

---
### 1. Module Overview
This notebook conducts initial dataset acquisition inspection, schema verification, data type analysis, missing value profiling, duplicate row identification, and infinite numerical value detection.

```
Defensive Principle: Reliable cybersecurity models require forensically sound,
well-understood network baseline data without silent errors.
```

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to Python module search path
sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
import numpy as np
from src.config import config
from src.data_loader import check_raw_dataset_status, load_dataset, inspect_dataset

print("Environment configured successfully.")

### 2. Dataset Acquisition & Raw File Verification
Checking whether raw CIC-IDS2017 CSV files have been downloaded to `data/raw/`.

In [ ]:
status = check_raw_dataset_status()
print("Raw Dataset Status:")
for k, v in status.items():
    print(f"  {k}: {v}")

### 3. Loading Dataset Records
Loading records with schema normalization and automatic fallback to verified software fixtures when raw files are being staged.

In [ ]:
df, metadata = load_dataset(use_sample_if_empty=True)
print(f"Loaded {len(df):,} records from source: {metadata.get('source')}")
print(f"Dataset Dimensions: {df.shape[0]:,} Rows × {df.shape[1]} Columns")
df.head(5)

### 4. Comprehensive Dataset Quality Inspection
Analyzing missing values, infinite values, duplicate rows, memory usage, and class distributions.

In [ ]:
summary = inspect_dataset(df)

print("DATA QUALITY AUDIT:")
print(f"  Total Rows: {summary['total_rows']:,}")
print(f"  Total Columns: {summary['total_columns']}")
print(f"  Memory Footprint: {summary['memory_usage_mb']:.2f} MB")
print(f"  Missing Values: {summary['total_missing_values']:,}")
print(f"  Infinite Values: {summary['total_infinite_values']:,}")
print(f"  Duplicate Rows: {summary['duplicate_rows']:,}")
print(f"  Benign Traffic Ratio: {summary['benign_count']:,} ({(summary['benign_count']/len(df))*100:.1f}%)")
print(f"  Malicious Traffic Ratio: {summary['attack_count']:,} ({summary['attack_percentage']:.1f}%)")
print("\nTraffic Class Frequencies:")
for lbl, cnt in summary['label_counts'].items():
    print(f"  - {lbl}: {cnt:,}")

### 5. Architectural Findings & Key Takeaways
1. **Inconsistent Column Headers:** Raw CIC-IDS2017 contains leading/trailing whitespaces in column names (e.g., `' Destination Port'`), requiring automated string trimming.
2. **Infinite Values:** Ratios such as `Flow Bytes/s` contain division-by-zero artifacts (`inf`), necessitating replacement with `NaN` and robust median imputation.
3. **Severe Class Imbalance:** Benign user traffic dominates the flow distribution (~70-80%), reflecting real-world enterprise environments where attacks are minority occurrences.